In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

In [5]:
os.chdir("/Users/kaiping/Desktop/olist_project/data") 
os.getcwd()

'/Users/kaiping/Desktop/olist_project/data'

In [6]:
df_translation = pd.read_csv("raw/product_category_name_translation.csv")
df_sellers     = pd.read_csv("raw/olist_sellers_dataset.csv")
df_products    = pd.read_csv("raw/olist_products_dataset.csv")
df_orders      = pd.read_csv("raw/olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv("raw/olist_order_reviews_dataset.csv")
df_order_payments = pd.read_csv("raw/olist_order_payments_dataset.csv")
df_order_items    = pd.read_csv("raw/olist_order_items_dataset.csv")
df_geolocation    = pd.read_csv("raw/olist_geolocation_dataset.csv")
df_customers      = pd.read_csv("raw/olist_customers_dataset.csv")


In [7]:


# =========================
# 0. 參數設定
# =========================
PRED_DAYS = 90
RANDOM_STATE = 42

# =========================
# 1. 基礎清理
# =========================
# copy 避免污染原始資料
translation = df_translation.copy()
sellers = df_sellers.copy()
products = df_products.copy()
orders = df_orders.copy()
order_reviews = df_order_reviews.copy()
order_payments = df_order_payments.copy()
order_items = df_order_items.copy()
geolocation = df_geolocation.copy()
customers = df_customers.copy()

# 時間欄位轉 datetime
order_dt_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in order_dt_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# 補商品英文類別
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# 顧客主鍵：customer_unique_id
orders = orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
    on="customer_id",
    how="left"
)

# 只保留 delivered 訂單，且至少要有 purchase + delivered date
orders_delivered = orders.loc[
    (orders["order_status"] == "delivered") &
    orders["order_purchase_timestamp"].notna() &
    orders["order_delivered_customer_date"].notna()
].copy()

print("delivered orders:", orders_delivered.shape)

# 觀測到的最後購買時間
data_end = orders_delivered["order_purchase_timestamp"].max()
analysis_end = data_end - pd.Timedelta(days=PRED_DAYS)

print("data_end:", data_end)
print("analysis_end:", analysis_end)

# 後面做 event 聚合會用到的 key
order_key = orders_delivered[[
    "order_id",
    "customer_unique_id",
    "customer_state",
    "customer_city",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].copy()

# =========================
# 2. 建立「購買事件」層級特徵
#    同 customer_unique_id + 同 order_purchase_timestamp
#    視為同一次 purchase event
# =========================

# ---------- 2.1 商品 / 金額特徵 ----------
item_base = order_items.merge(
    order_key[["order_id", "customer_unique_id", "order_purchase_timestamp"]],
    on="order_id",
    how="inner"
)

item_base = item_base.merge(
    products[[
        "product_id",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_photos_qty",
        "product_name_lenght",
        "product_description_lenght"
    ]],
    on="product_id",
    how="left"
)

item_base["gmv_item"] = item_base["price"] + item_base["freight_value"]
item_base["product_volume_cm3"] = (
    item_base["product_length_cm"] *
    item_base["product_height_cm"] *
    item_base["product_width_cm"]
)

def mode_or_nan(x):
    x = x.dropna()
    if x.empty:
        return np.nan
    return x.mode().iloc[0]

event_item_agg = (
    item_base
    .groupby(["customer_unique_id", "order_purchase_timestamp"], as_index=False)
    .agg(
        purchase_event_order_cnt=("order_id", "nunique"),
        first_event_item_cnt=("order_item_id", "count"),
        first_event_product_cnt=("product_id", "nunique"),
        first_event_seller_cnt=("seller_id", "nunique"),
        first_event_category_cnt=("product_category_name_english", "nunique"),
        first_event_gmv=("gmv_item", "sum"),
        first_event_price_sum=("price", "sum"),
        first_event_freight_sum=("freight_value", "sum"),
        first_event_avg_item_price=("price", "mean"),
        first_event_max_item_price=("price", "max"),
        first_event_avg_weight_g=("product_weight_g", "mean"),
        first_event_avg_volume_cm3=("product_volume_cm3", "mean"),
        first_event_avg_photos_qty=("product_photos_qty", "mean"),
        first_event_avg_name_len=("product_name_lenght", "mean"),
        first_event_avg_desc_len=("product_description_lenght", "mean"),
        first_event_dom_category=("product_category_name_english", mode_or_nan)
    )
)

# ---------- 2.2 付款特徵 ----------
payment_base = order_payments.merge(
    order_key[["order_id", "customer_unique_id", "order_purchase_timestamp"]],
    on="order_id",
    how="inner"
)

event_payment_agg = (
    payment_base
    .groupby(["customer_unique_id", "order_purchase_timestamp"], as_index=False)
    .agg(
        first_event_payment_cnt=("payment_sequential", "count"),
        first_event_payment_type_nunique=("payment_type", "nunique"),
        first_event_payment_type_mode=("payment_type", mode_or_nan),
        first_event_installments_max=("payment_installments", "max"),
        first_event_installments_mean=("payment_installments", "mean"),
        first_event_payment_value_sum=("payment_value", "sum")
    )
)

# ---------- 2.3 評價特徵 ----------
review_base = order_reviews.merge(
    order_key[["order_id", "customer_unique_id", "order_purchase_timestamp"]],
    on="order_id",
    how="inner"
)

review_base["low_review_flag"] = (review_base["review_score"] <= 3).astype(int)

event_review_agg = (
    review_base
    .groupby(["customer_unique_id", "order_purchase_timestamp"], as_index=False)
    .agg(
        first_event_review_cnt=("review_id", "count"),
        first_event_review_score_mean=("review_score", "mean"),
        first_event_review_score_min=("review_score", "min"),
        first_event_low_review_rate=("low_review_flag", "mean")
    )
)

# ---------- 2.4 物流 / 時間體驗特徵 ----------
order_delivery_base = order_key.copy()

order_delivery_base["approval_lag_days"] = (
    order_delivery_base["order_approved_at"] - order_delivery_base["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_delivery_base["purchase_to_carrier_days"] = (
    order_delivery_base["order_delivered_carrier_date"] - order_delivery_base["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_delivery_base["purchase_to_delivered_days"] = (
    order_delivery_base["order_delivered_customer_date"] - order_delivery_base["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_delivery_base["carrier_to_customer_days"] = (
    order_delivery_base["order_delivered_customer_date"] - order_delivery_base["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

order_delivery_base["delivery_delay_vs_estimated_days"] = (
    order_delivery_base["order_delivered_customer_date"] - order_delivery_base["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

order_delivery_base["late_delivery_flag"] = (
    order_delivery_base["delivery_delay_vs_estimated_days"] > 0
).astype(int)

event_delivery_agg = (
    order_delivery_base
    .groupby(["customer_unique_id", "order_purchase_timestamp"], as_index=False)
    .agg(
        first_event_delivered_date=("order_delivered_customer_date", "max"),   # snapshot 用 max，比較保守
        first_event_customer_state=("customer_state", mode_or_nan),
        first_event_customer_city=("customer_city", mode_or_nan),
        first_event_approval_lag_days_mean=("approval_lag_days", "mean"),
        first_event_purchase_to_carrier_days_mean=("purchase_to_carrier_days", "mean"),
        first_event_purchase_to_delivered_days_mean=("purchase_to_delivered_days", "mean"),
        first_event_carrier_to_customer_days_mean=("carrier_to_customer_days", "mean"),
        first_event_delay_vs_estimated_days_mean=("delivery_delay_vs_estimated_days", "mean"),
        first_event_late_delivery_rate=("late_delivery_flag", "mean")
    )
)

# ---------- 2.5 合併 event 特徵 ----------
event_df = event_delivery_agg.merge(
    event_item_agg,
    on=["customer_unique_id", "order_purchase_timestamp"],
    how="left"
).merge(
    event_payment_agg,
    on=["customer_unique_id", "order_purchase_timestamp"],
    how="left"
).merge(
    event_review_agg,
    on=["customer_unique_id", "order_purchase_timestamp"],
    how="left"
)

# 派生欄位
event_df["first_event_freight_ratio"] = np.where(
    event_df["first_event_gmv"] > 0,
    event_df["first_event_freight_sum"] / event_df["first_event_gmv"],
    np.nan
)

event_df["purchase_month"] = event_df["order_purchase_timestamp"].dt.month.astype("Int64").astype(str)
event_df["purchase_dow"] = event_df["order_purchase_timestamp"].dt.dayofweek.astype("Int64").astype(str)
event_df["purchase_hour"] = event_df["order_purchase_timestamp"].dt.hour

def hour_bin(h):
    if pd.isna(h):
        return np.nan
    if 0 <= h < 6:
        return "night"
    elif 6 <= h < 12:
        return "morning"
    elif 12 <= h < 18:
        return "afternoon"
    else:
        return "evening"

event_df["purchase_hour_bin"] = event_df["purchase_hour"].apply(hour_bin)

# review 缺失旗標
event_df["has_review"] = event_df["first_event_review_cnt"].fillna(0).gt(0).astype(int)

# =========================
# 3. 取每位顧客第一個購買事件
# =========================
event_df = event_df.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).reset_index(drop=True)

event_df["event_rank"] = event_df.groupby("customer_unique_id").cumcount() + 1
event_df["next_event_purchase_ts"] = event_df.groupby("customer_unique_id")["order_purchase_timestamp"].shift(-1)

first_event_df = event_df.loc[event_df["event_rank"] == 1].copy()

# =========================
# 4. 建 label
#    預測時點 = first_event_delivered_date
#    標籤 = delivered 後 90 天內是否有第二次購買事件
# =========================

# 只保留 snapshot 有完整90天追蹤窗者
first_event_df = first_event_df.loc[
    first_event_df["first_event_delivered_date"].notna() &
    (first_event_df["first_event_delivered_date"] <= analysis_end)
].copy()

# 排除在 first_event_delivered_date 前就已經發生第二次購買的人
# 因為到 snapshot 時，他已經不是「首購後待轉化」對象了
first_event_df = first_event_df.loc[
    first_event_df["next_event_purchase_ts"].isna() |
    (first_event_df["next_event_purchase_ts"] > first_event_df["first_event_delivered_date"])
].copy()

first_event_df["purchase_2nd_within_90d"] = (
    first_event_df["next_event_purchase_ts"].notna() &
    (first_event_df["next_event_purchase_ts"] <= first_event_df["first_event_delivered_date"] + pd.Timedelta(days=PRED_DAYS))
).astype(int)

print("model sample size:", first_event_df.shape)
print("positive rate:", first_event_df["purchase_2nd_within_90d"].mean())

# =========================
# 5. 建模欄位整理
# =========================
model_df = first_event_df.copy()

# 你可以先用這批變數跑第一版
numeric_features = [
    "purchase_event_order_cnt",
    "first_event_item_cnt",
    "first_event_product_cnt",
    "first_event_seller_cnt",
    "first_event_category_cnt",
    "first_event_gmv",
    "first_event_price_sum",
    "first_event_freight_sum",
    "first_event_freight_ratio",
    "first_event_avg_item_price",
    "first_event_max_item_price",
    "first_event_avg_weight_g",
    "first_event_avg_volume_cm3",
    "first_event_avg_photos_qty",
    "first_event_avg_name_len",
    "first_event_avg_desc_len",
    "first_event_payment_cnt",
    "first_event_payment_type_nunique",
    "first_event_installments_max",
    "first_event_installments_mean",
    "first_event_payment_value_sum",
    "first_event_review_cnt",
    "first_event_review_score_mean",
    "first_event_review_score_min",
    "first_event_low_review_rate",
    "first_event_approval_lag_days_mean",
    "first_event_purchase_to_carrier_days_mean",
    "first_event_purchase_to_delivered_days_mean",
    "first_event_carrier_to_customer_days_mean",
    "first_event_delay_vs_estimated_days_mean",
    "first_event_late_delivery_rate",
    "has_review"
]

categorical_features = [
    "first_event_customer_state",
    "first_event_dom_category",
    "first_event_payment_type_mode",
    "purchase_month",
    "purchase_dow",
    "purchase_hour_bin"
]

target_col = "purchase_2nd_within_90d"

keep_cols = ["customer_unique_id", "order_purchase_timestamp", "first_event_delivered_date", target_col] + numeric_features + categorical_features
model_df = model_df[keep_cols].copy()

# 類別過多時，可先把稀有類別合併
def collapse_rare_categories(series, min_count=100):
    vc = series.value_counts(dropna=False)
    valid = vc[vc >= min_count].index
    return series.where(series.isin(valid), other="OTHER")

for col in ["first_event_dom_category"]:
    model_df[col] = collapse_rare_categories(model_df[col], min_count=100)

# 目標與特徵
X = model_df[numeric_features + categorical_features].copy()
y = model_df[target_col].copy()

# =========================
# 6. 時間切分
#    用首購 snapshot 時間做排序切分
# =========================
split_df = model_df[["first_event_delivered_date"]].copy().sort_values("first_event_delivered_date")
cutoff = split_df["first_event_delivered_date"].quantile(0.8)

train_idx = model_df["first_event_delivered_date"] <= cutoff
test_idx = model_df["first_event_delivered_date"] > cutoff

X_train = X.loc[train_idx].copy()
X_test = X.loc[test_idx].copy()
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

print("train size:", X_train.shape, "test size:", X_test.shape)
print("train pos rate:", y_train.mean())
print("test pos rate:", y_test.mean())

# =========================
# 7. Preprocess + Logistic Regression
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logit_model.fit(X_train, y_train)

# =========================
# 8. 模型評估
# =========================
y_pred = logit_model.predict(X_test)
y_proba = logit_model.predict_proba(X_test)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))

print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("PR AUC :", average_precision_score(y_test, y_proba))

# Top 10% precision / lift
test_result = X_test.copy()
test_result["y_true"] = y_test.values
test_result["score"] = y_proba
test_result = test_result.sort_values("score", ascending=False).reset_index(drop=True)

top_k = max(1, int(len(test_result) * 0.10))
top10 = test_result.head(top_k)

base_rate = test_result["y_true"].mean()
top10_precision = top10["y_true"].mean()
top10_lift = top10_precision / base_rate if base_rate > 0 else np.nan

print("\n=== Ranking Metrics ===")
print("Base rate:", base_rate)
print("Top 10% precision:", top10_precision)
print("Top 10% lift:", top10_lift)

# =========================
# 9. 抽出 Logistic Regression 係數
# =========================
# 先拿 preprocessor 與 model
fitted_preprocessor = logit_model.named_steps["preprocessor"]
fitted_lr = logit_model.named_steps["model"]

# 取得展開後欄位名
feature_names = fitted_preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": fitted_lr.coef_[0]
})

coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("\n=== Top 30 coefficients by absolute value ===")
print(coef_df.head(30))

# =========================
# 10. 讓結果更好讀：拆成正向 / 負向影響
# =========================
positive_coef_df = coef_df.sort_values("coef", ascending=False).reset_index(drop=True)
negative_coef_df = coef_df.sort_values("coef", ascending=True).reset_index(drop=True)

print("\n=== Top 20 positive drivers (較可能二購) ===")
print(positive_coef_df.head(20))

print("\n=== Top 20 negative drivers (較不可能二購) ===")
print(negative_coef_df.head(20))

# =========================
# 11. 輸出可檢查的建模資料與名單
# =========================
model_output = model_df.loc[test_idx, ["customer_unique_id", "order_purchase_timestamp", "first_event_delivered_date"]].copy()
model_output["y_true"] = y_test.values
model_output["score"] = y_proba
model_output = model_output.sort_values("score", ascending=False).reset_index(drop=True)

print("\n=== Top 20 high propensity customers ===")
print(model_output.head(20))

# 如要存檔可取消註解
# model_df.to_csv("interim/second_purchase_model_df.csv", index=False)
# coef_df.to_csv("interim/logit_coef_df.csv", index=False)
# model_output.to_csv("interim/second_purchase_scored_test.csv", index=False)

delivered orders: (96470, 11)
data_end: 2018-08-29 15:00:00
analysis_end: 2018-05-31 15:00:00
model sample size: (72706, 46)
positive rate: 0.011154512694963276
train size: (58165, 38) test size: (14541, 38)
train pos rate: 0.010968795667497635
test pos rate: 0.01189739357678289

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9913    0.5219    0.6837     14368
           1     0.0153    0.6185    0.0299       173

    accuracy                         0.5230     14541
   macro avg     0.5033    0.5702    0.3568     14541
weighted avg     0.9797    0.5230    0.6760     14541

ROC AUC: 0.5973804182705305
PR AUC : 0.019157587786602984

=== Ranking Metrics ===
Base rate: 0.01189739357678289
Top 10% precision: 0.021320495185694635
Top 10% lift: 1.7920307543074316

=== Top 30 coefficients by absolute value ===
                                              feature      coef  abs_coef
0                  cat__first_event_customer_state_TO 

### 2. 第二版精簡變數模型

In [8]:

# ======================================
# A. 以 first_event_df 為起點
#    假設你前面已完成：
#    1. event_df 建立
#    2. first_event_df 建立
#    3. purchase_2nd_within_90d 標籤建立
# ======================================

model_df_v2 = first_event_df.copy()

print("model_df_v2 shape:", model_df_v2.shape)
print("positive rate:", model_df_v2["purchase_2nd_within_90d"].mean())

# ======================================
# B. 精簡特徵：只留較穩定、較可解釋欄位
# ======================================

# 數值欄位：刪除共線 / 噪音較高欄位後保留
numeric_features_v2 = [
    # 首購交易強度
    "first_event_gmv",
    "first_event_item_cnt",
    "first_event_product_cnt",
    "first_event_seller_cnt",
    "first_event_category_cnt",
    "first_event_freight_ratio",

    # 首購付款
    "first_event_installments_max",

    # 首購體驗
    "has_review",
    "first_event_review_score_mean",
    "first_event_low_review_rate",
    "first_event_purchase_to_delivered_days_mean",
    "first_event_delay_vs_estimated_days_mean",
    "first_event_late_delivery_rate"
]

# 類別欄位：保留州別 / 主品類 / 支付方式 / 時間
categorical_features_v2 = [
    "first_event_customer_state",
    "first_event_dom_category",
    "first_event_payment_type_mode",
    "purchase_month",
    "purchase_dow",
    "purchase_hour_bin"
]

target_col = "purchase_2nd_within_90d"

keep_cols_v2 = (
    ["customer_unique_id", "order_purchase_timestamp", "first_event_delivered_date", target_col]
    + numeric_features_v2
    + categorical_features_v2
)

model_df_v2 = model_df_v2[keep_cols_v2].copy()

# ======================================
# C. 類別合併：只保留高頻類別，其餘歸 OTHER
# ======================================
def keep_top_categories(series, top_n=15):
    top_values = series.value_counts(dropna=False).head(top_n).index
    return series.where(series.isin(top_values), other="OTHER")

# 州別：保留前10大
model_df_v2["first_event_customer_state"] = keep_top_categories(
    model_df_v2["first_event_customer_state"], top_n=10
)

# 品類：保留前15大
model_df_v2["first_event_dom_category"] = keep_top_categories(
    model_df_v2["first_event_dom_category"], top_n=15
)

# payment type 通常類別不多，但還是保底一下
model_df_v2["first_event_payment_type_mode"] = keep_top_categories(
    model_df_v2["first_event_payment_type_mode"], top_n=5
)

# 月份 / 星期 / 時段轉字串
for col in ["purchase_month", "purchase_dow", "purchase_hour_bin"]:
    model_df_v2[col] = model_df_v2[col].astype("string")

# ======================================
# D. 時間切分
#    仍然用 first_event_delivered_date 切 train/test
# ======================================
cutoff_v2 = model_df_v2["first_event_delivered_date"].quantile(0.8)

train_idx_v2 = model_df_v2["first_event_delivered_date"] <= cutoff_v2
test_idx_v2 = model_df_v2["first_event_delivered_date"] > cutoff_v2

X_train_v2 = model_df_v2.loc[train_idx_v2, numeric_features_v2 + categorical_features_v2].copy()
X_test_v2  = model_df_v2.loc[test_idx_v2, numeric_features_v2 + categorical_features_v2].copy()

y_train_v2 = model_df_v2.loc[train_idx_v2, target_col].copy()
y_test_v2  = model_df_v2.loc[test_idx_v2, target_col].copy()

print("train size:", X_train_v2.shape)
print("test size :", X_test_v2.shape)
print("train pos rate:", y_train_v2.mean())
print("test pos rate :", y_test_v2.mean())

# ======================================
# E. Preprocessing + Logistic Regression
# ======================================
numeric_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor_v2 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_v2, numeric_features_v2),
        ("cat", categorical_transformer_v2, categorical_features_v2)
    ]
)

logit_v2 = Pipeline(steps=[
    ("preprocessor", preprocessor_v2),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

logit_v2.fit(X_train_v2, y_train_v2)

# ======================================
# F. 預測
# ======================================
y_pred_v2 = logit_v2.predict(X_test_v2)
y_proba_v2 = logit_v2.predict_proba(X_test_v2)[:, 1]

print("\n=== Classification Report (v2) ===")
print(classification_report(y_test_v2, y_pred_v2, digits=4))

print("ROC AUC:", roc_auc_score(y_test_v2, y_proba_v2))
print("PR AUC :", average_precision_score(y_test_v2, y_proba_v2))

# ======================================
# G. Ranking 指標
# ======================================
ranking_df_v2 = model_df_v2.loc[test_idx_v2, [
    "customer_unique_id",
    "order_purchase_timestamp",
    "first_event_delivered_date"
]].copy()

ranking_df_v2["y_true"] = y_test_v2.values
ranking_df_v2["score"] = y_proba_v2
ranking_df_v2 = ranking_df_v2.sort_values("score", ascending=False).reset_index(drop=True)

base_rate_v2 = ranking_df_v2["y_true"].mean()

def top_k_metrics(df, top_pct):
    top_k = max(1, int(len(df) * top_pct))
    top_df = df.head(top_k)
    precision = top_df["y_true"].mean()
    lift = precision / base_rate_v2 if base_rate_v2 > 0 else np.nan
    capture = top_df["y_true"].sum() / df["y_true"].sum() if df["y_true"].sum() > 0 else np.nan
    return {
        "top_pct": top_pct,
        "top_k": top_k,
        "precision": precision,
        "lift": lift,
        "capture_rate": capture
    }

metrics_5 = top_k_metrics(ranking_df_v2, 0.05)
metrics_10 = top_k_metrics(ranking_df_v2, 0.10)
metrics_20 = top_k_metrics(ranking_df_v2, 0.20)

ranking_metrics_v2 = pd.DataFrame([metrics_5, metrics_10, metrics_20])

print("\n=== Ranking Metrics (v2) ===")
print("Base rate:", base_rate_v2)
print(ranking_metrics_v2)

# ======================================
# H. Decile lift
# ======================================
ranking_df_v2["decile"] = pd.qcut(
    ranking_df_v2["score"].rank(method="first", ascending=False),
    10,
    labels=[1,2,3,4,5,6,7,8,9,10]
)

decile_summary_v2 = (
    ranking_df_v2
    .groupby("decile", as_index=False)
    .agg(
        customer_cnt=("customer_unique_id", "count"),
        positive_cnt=("y_true", "sum"),
        avg_score=("score", "mean"),
        precision=("y_true", "mean")
    )
)

decile_summary_v2["lift"] = decile_summary_v2["precision"] / base_rate_v2

print("\n=== Decile Summary (v2) ===")
print(decile_summary_v2)

# ======================================
# I. Logistic Regression 權重
# ======================================
fitted_preprocessor_v2 = logit_v2.named_steps["preprocessor"]
fitted_lr_v2 = logit_v2.named_steps["model"]

feature_names_v2 = fitted_preprocessor_v2.get_feature_names_out()

coef_df_v2 = pd.DataFrame({
    "feature": feature_names_v2,
    "coef": fitted_lr_v2.coef_[0]
})

coef_df_v2["abs_coef"] = coef_df_v2["coef"].abs()
coef_df_v2 = coef_df_v2.sort_values("abs_coef", ascending=False).reset_index(drop=True)

positive_coef_df_v2 = coef_df_v2.sort_values("coef", ascending=False).reset_index(drop=True)
negative_coef_df_v2 = coef_df_v2.sort_values("coef", ascending=True).reset_index(drop=True)

print("\n=== Top 20 coefficients by abs value (v2) ===")
print(coef_df_v2.head(20))

print("\n=== Top 15 positive drivers (v2) ===")
print(positive_coef_df_v2.head(15))

print("\n=== Top 15 negative drivers (v2) ===")
print(negative_coef_df_v2.head(15))

# ======================================
# J. 輸出高分名單
# ======================================
top_scored_v2 = ranking_df_v2.sort_values("score", ascending=False).reset_index(drop=True)

print("\n=== Top 20 high propensity customers (v2) ===")
print(top_scored_v2.head(20))

# ======================================
# K. 存檔（可選）
# ======================================
# model_df_v2.to_csv("interim/second_purchase_model_df_v2.csv", index=False)
# ranking_metrics_v2.to_csv("interim/second_purchase_ranking_metrics_v2.csv", index=False)
# decile_summary_v2.to_csv("interim/second_purchase_decile_summary_v2.csv", index=False)
# coef_df_v2.to_csv("interim/logit_coef_df_v2.csv", index=False)
# top_scored_v2.to_csv("interim/second_purchase_scored_test_v2.csv", index=False)

model_df_v2 shape: (72706, 46)
positive rate: 0.011154512694963276
train size: (58165, 19)
test size : (14541, 19)
train pos rate: 0.010968795667497635
test pos rate : 0.01189739357678289

=== Classification Report (v2) ===
              precision    recall  f1-score   support

           0     0.9905    0.5226    0.6842     14368
           1     0.0145    0.5838    0.0283       173

    accuracy                         0.5233     14541
   macro avg     0.5025    0.5532    0.3562     14541
weighted avg     0.9789    0.5233    0.6764     14541

ROC AUC: 0.5965049982620338
PR AUC : 0.02230650872621717

=== Ranking Metrics (v2) ===
Base rate: 0.01189739357678289
   top_pct  top_k  precision      lift  capture_rate
0     0.05    727   0.024759  2.081068      0.104046
1     0.10   1454   0.021320  1.792031      0.179191
2     0.20   2908   0.019257  1.618608      0.323699

=== Decile Summary (v2) ===
  decile  customer_cnt  positive_cnt  avg_score  precision      lift
0      1          145

/var/folders/vl/wskb2r0j55g_xcyvprs302bc0000gn/T/ipykernel_21379/2230200698.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ranking_df_v2


In [9]:
corr_check_cols = [
    "first_event_gmv",
    "first_event_item_cnt",
    "first_event_product_cnt",
    "first_event_seller_cnt",
    "first_event_category_cnt",
    "first_event_freight_ratio",
    "first_event_installments_max",
    "has_review",
    "first_event_review_score_mean",
    "first_event_low_review_rate",
    "first_event_purchase_to_delivered_days_mean",
    "first_event_delay_vs_estimated_days_mean",
    "first_event_late_delivery_rate"
]

corr_matrix_v2 = model_df_v2[corr_check_cols].corr(numeric_only=True)
print(corr_matrix_v2.round(3))

                                             first_event_gmv  \
first_event_gmv                                        1.000   
first_event_item_cnt                                   0.199   
first_event_product_cnt                                0.088   
first_event_seller_cnt                                 0.064   
first_event_category_cnt                               0.035   
first_event_freight_ratio                             -0.394   
first_event_installments_max                           0.323   
has_review                                            -0.012   
first_event_review_score_mean                         -0.043   
first_event_low_review_rate                            0.033   
first_event_purchase_to_delivered_days_mean            0.070   
first_event_delay_vs_estimated_days_mean              -0.006   
first_event_late_delivery_rate                         0.023   

                                             first_event_item_cnt  \
first_event_gmv                   

### 3. 對比random forest 和邏輯斯回歸